# lmcircuit

Ask judges one at a time, and stop when the answer is worth its cost. This notebook:

1. **gets a dataset**: either one of the registered benchmarks (and inspects it), or your own questions with ground
   truth, answered by models you choose;
2. **evaluates** circuits optimised for accuracy and for balanced accuracy, against each judge alone and, optionally, eight published baselines; every circuit is scored on every metric;
3. **uses** one fitted circuit item by item.

A GPU makes step 2 much faster. Run either 1a or 1b, then the rest.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import lmcircuit as lc

DATA = Path("data")          # raw files of the registered datasets: DATA/<name>
OUT = Path("runs")           # datasets, reports and figures: OUT/<name>

## 1a. A registered dataset

`lc.datasets.REGISTRY` lists them; each loader's docstring gives its source, licence, items, judges and costs.
`fetch` downloads the public sources once (it needs network access).

In [ ]:
print(lc.datasets.REGISTRY)
NAME = "oll_bbh"
# lc.datasets.fetch(NAME, DATA / NAME)          # once
ds = lc.datasets.load(NAME, DATA / NAME)

In [ ]:
def describe(ds: lc.Dataset):
    """How much data, the label balance, tasks and groups, and one row per judge."""
    lab = ds.labels[ds.labelled]
    shares = lab.sum(axis=0) / lab.sum()
    print(f"{ds.n_items:,} items, {len(set(ds.task))} tasks, {len(set(ds.group)):,} groups, "
          f"at most {ds.answers.shape[2]} recorded calls per judge and item")
    print("classes:", ", ".join(f"{c} {100 * s:.0f}%" for c, s in zip(ds.classes or range(ds.n_classes), shares)))
    truth = lab.argmax(axis=1)
    rows = []
    for j, judge in enumerate(ds.judges):
        a, ok = ds.answers[ds.labelled, j, 0], ds.answers[ds.labelled, j, 0] >= 0
        rows.append({"judge": judge.name, "model": judge.model,
                     "cost per call (priciest = 1)": np.nanmean(np.where(ds.valid[:, j], ds.cost[:, j], np.nan)) / ds.scale(),
                     "accuracy": (a[ok] == truth[ok]).mean(), "answered": ok.mean()})
    return pd.DataFrame(rows).sort_values("cost per call (priciest = 1)").round(3)

describe(ds)

In [ ]:
# a few items: what the prior reads (the item, as the judges see it), the label, and every judge's first answer
for i in np.random.default_rng(0).choice(ds.n_items, 3, replace=False):
    print(f"--- {ds.ids[i]} (task {ds.task[i]}), label: {ds.classes[ds.labels[i].argmax()] if ds.classes else ds.labels[i].argmax()}")
    print(str(ds.prior_text["item"][i])[:600])
    print({j.name: int(ds.answers[i, k, 0]) for k, j in enumerate(ds.judges)})

## 1b. Or your own task

Questions with their ground truth, the classes, and the models to choose between. Each model is a Hugging Face model
served by vLLM (`vllm serve <model>`) or any OpenAI-compatible endpoint, under a system prompt: the shared one, or its
own. The same model may appear twice with different system prompts. Each judge answers every question `SAMPLES` times;
the answer is the class it names last.

In [ ]:
NAME = "my_task"
TASK = "Decide whether the statement is true."
CLASSES = ["false", "true"]
QUESTIONS = pd.DataFrame({"question": ["The Earth orbits the Sun.", "Water boils at 50 C at sea level."],
                          "label": [1, 0]})        # or pd.read_csv(...): one row per question, label = class index
# (two rows only show the format: evaluation needs at least a few hundred questions, over 5 grouped folds)
SYSTEM = "Answer with one word: true or false."
MODELS = [   # name, model, server, USD per million input / output tokens, and optionally its own system prompt
    {"name": "qwen3-4b", "model": "Qwen/Qwen3-4B", "url": "http://localhost:8001/v1", "price": (0.03, 0.03)},
    {"name": "qwen3-32b", "model": "Qwen/Qwen3-32B", "url": "http://localhost:8002/v1", "price": (0.08, 0.28)},
    {"name": "qwen3-32b-careful", "model": "Qwen/Qwen3-32B", "url": "http://localhost:8002/v1", "price": (0.08, 0.28),
     "system": "Think about the statement carefully, then answer with one word: true or false."},
]
SAMPLES = 3

items = [{"id": str(i), "item": q, "task_description": TASK} for i, q in enumerate(QUESTIONS["question"])]
ds = lc.Dataset.from_records(items, n_classes=len(CLASSES), classes=CLASSES, labels=QUESTIONS["label"].to_numpy())
judges = [lc.JudgeSpec(m["name"], lc.Backend.vllm(m["url"], m["model"], price_in=m["price"][0], price_out=m["price"][1]),
                       lc.Prompt("{item}", CLASSES, system=m.get("system", SYSTEM))) for m in MODELS]
ds = lc.collect(ds, judges, repeats=SAMPLES)        # answers, USD per call, planning cost; cached
describe(ds)

## 2. Evaluate

The prior's features embed each item's text, its task description then the item (`lc.embed`). `lc.evaluate` then cross-fits over five grouped folds and
five training seeds: every item is tested once, by circuits fitted on the other folds. Uncomment `baselines` to add the
eight published methods (the two fine-tuned ones need the `baselines` extra and take hours on large datasets).

In [ ]:
ds = lc.embed(ds)
ds.save(OUT / NAME / "dataset")
report = lc.evaluate(ds, out=OUT / NAME / "run")    # every (fold, seed) saved; report.json too

In [ ]:
# frontier areas (each metric averaged over budgets): one row per objective the circuit optimised
pd.DataFrame({o: r["circuit"]["area"] for o, r in report["objectives"].items()}).T.mul(100).round(1)

In [ ]:
from IPython.display import IFrame
figures = lc.plot.figures(report, OUT / NAME / "figures", title=NAME)
print(sorted(p.name for p in figures.glob("*.pdf")))
IFrame(str(figures / "accuracy.pdf"), width=900, height=620)

## 3. Use one circuit

Fit on one split, choose a budget, save, and decide item by item. Here a judge's answer is the recorded one; in a
live system `ask` calls the model.

In [ ]:
train, val, test = next(ds.folds(5))
circuit = lc.Circuit().fit(ds, train, val)              # objective: accuracy (circuit.objective(...) to switch)
point = circuit.at_budget(0.05)                          # spend at most 5% of the priciest judge per item
circuit.save(OUT / NAME / "circuit")

def ask(judge, item):
    return int(ds.answers[item, ds.judge_names.index(judge), 0])

for item in test[:5]:
    policy, answers = circuit.policy(ds.features[item], point), {}
    while (decision := policy.decide(answers)).ask is not None:
        answers[decision.ask] = ask(decision.ask, item)
    print(f"{ds.ids[item]}: asked {list(answers) or 'nobody'}, predicted {decision.predict}, "
          f"belief {np.round(decision.belief, 2)}, label {ds.labels[item].argmax()}")